# Distant Listening Corpus `chords` facet → BOPP `ScoreObject`

The `chords.tsv` facet is heterogeneous: every row is one of seven event types (`Chord`, `Dynamic`, `Spanner`,
`FiguredBass`, `StaffText`, `SystemText`, `Tempo`), each populating a different subset of the 77 columns.

Proposal: a top-level **`Object`** union (`schemas/v1/object.json`, discriminated by `object_type`) with two members:

* **`Annotation`** – the existing single-timeline object: one extent + one payload (+ confidence).
* **`ScoreObject`** – one media, one timeline, *many* `EventBlock`s. Each block is a homogeneous
  extent/payload pair with its own strictly typed payload schema (`score_chord`, `score_dynamic`, …), yet all
  blocks live in the same msgpack file.

A tabular export is the outer union of the blocks: a **sparse table** in which only the columns of a row's payload
type are filled, plus a `payload_type` column that makes the CSV self-describing (and reloadable).

In [1]:
from pathlib import Path

import pandas as pd

from bopp.io import load_bopp_msgpack, read_bopp_csv
from bopp.util import to_dataframe
from scripts.dlc_to_bopp import FACETS, export_facet, load_dlc, struct_columns

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'dlc_bopp_exports' / 'chords'
spec = FACETS['chords']

C:\Users\hentschel\git\bopp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## One payload schema per event type

Columns are taken from the generated msgspec models, i.e. from `schemas/v1/payload/score_*.json`. Shared score columns (measure, onset, staff, voice, duration, …) recur in every schema; `quarterbeats`, `mc_onset`, `mn_onset`, `duration` are `FractionBuffer`s.

In [2]:
for event, payload_type in spec.event_payloads.items():
    print(f'{event:>12} -> {payload_type:<20} {struct_columns(payload_type)}')

       Chord -> score_chord          ['mc', 'mn', 'quarterbeats', 'mc_onset', 'mn_onset', 'timesig', 'staff', 'voice', 'duration', 'volta', 'slur', 'gracenote', 'nominal_duration', 'scalar', 'chord_id', 'articulation', 'crescendo_hairpin', 'decrescendo_hairpin', 'diminuendo_line', 'crescendo_line', 'tremolo', 'lyrics_1', 'lyrics_2', 'lyrics_3']
     Dynamic -> score_dynamic        ['mc', 'mn', 'quarterbeats', 'mc_onset', 'mn_onset', 'timesig', 'staff', 'voice', 'duration', 'volta', 'slur', 'dynamics', 'crescendo_hairpin', 'decrescendo_hairpin']
     Spanner -> score_spanner        ['mc', 'mn', 'quarterbeats', 'mc_onset', 'mn_onset', 'timesig', 'staff', 'voice', 'duration', 'volta', 'slur', 'crescendo_hairpin', 'decrescendo_hairpin', 'diminuendo_line', 'crescendo_line']
 FiguredBass -> score_figured_bass   ['mc', 'mn', 'quarterbeats', 'mc_onset', 'mn_onset', 'timesig', 'staff', 'voice', 'duration', 'volta', 'slur', 'thoroughbass_duration', 'thoroughbass_level_1', 'thoroughbass_level_2',

## Export

One `ScoreObject` per piece, in msgpack, JSON and CSV. `max_pieces` limits the run; drop it to export the whole corpus (1.58 M events).

In [3]:
sizes = export_facet(ROOT, 'chords', OUTPUT_DIR, max_pieces=12, dlc=load_dlc(ROOT))
summary = sizes.groupby('format', as_index=False).agg(files=('bytes', 'size'), observations=('observations', 'sum'), total_kib=('kib', 'sum'), bytes_per_observation=('bytes_per_observation', 'mean'))
summary

Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_01.bopp.msgpack (273768 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_01.bopp.json (594709 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_02.bopp.msgpack (175474 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_02.bopp.json (393749 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_03.bopp.msgpack (76727 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_03.bopp.json (165629 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_04.bopp.msgpack (336147 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n01op18-1_04.bopp.json (742075 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n02op18-2_01.bopp.msgpack (204667 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n02op18-2_01.bopp.json (448521 bytes)


...

Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_01.bopp.msgpack (197870 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_01.bopp.json (435836 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_02.bopp.msgpack (180193 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_02.bopp.json (410742 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_03.bopp.msgpack (81624 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_03.bopp.json (178034 bytes)


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_04.bopp.msgpack (276675 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\chords\ABC__n03op18-3_04.bopp.json (591832 bytes)


,format,files,observations,total_kib,bytes_per_observation
0,csv,12,40900,5129.276367,128.869919
1,json,12,40900,4800.447266,119.612337
2,msgpack,12,40900,2190.572266,54.499135


## Inside the msgpack file

The decoded object is a `ScoreObject` whose blocks are typed per event; every block's extent is a `time_interval` in exact `quarters`.

In [4]:
path = sorted(OUTPUT_DIR.glob('*.bopp.msgpack'))[0]
obj = load_bopp_msgpack(path)
print(type(obj).__name__, obj.media_id)
for block in obj.events:
    start = block.extent.start
    print(f'  {type(block.payload).__name__:<24} n={len(start.values.numerator):>5}  unit={type(start).__name__}')

ScoreObject dlc:ABC/n01op18-1_01
  ScoreChordPayload        n= 4444  unit=QuartersCoordinate
  ScoreDynamicPayload      n=  378  unit=QuartersCoordinate
  ScoreSpannerPayload      n=   64  unit=QuartersCoordinate
  ScoreStaffTextPayload    n=   71  unit=QuartersCoordinate


## The sparse table

`to_dataframe` unions the blocks and sorts by start position; the same table is what `to_csv` wrote.

In [5]:
df = to_dataframe(obj)
print(df.shape)
pd.set_option('display.max_columns', 30)
df.head(12)

(4957, 68)


,payload_type,extent:time_interval:start:quarters,extent:time_interval:duration:quarters,payload:score_chord:mc,payload:score_chord:mn,payload:score_chord:quarterbeats,payload:score_chord:mc_onset,payload:score_chord:mn_onset,payload:score_chord:timesig,payload:score_chord:staff,payload:score_chord:voice,payload:score_chord:duration,payload:score_chord:volta,payload:score_chord:slur,payload:score_chord:gracenote,...,payload:score_spanner:decrescendo_hairpin,payload:score_spanner:diminuendo_line,payload:score_spanner:crescendo_line,payload:score_staff_text:mc,payload:score_staff_text:mn,payload:score_staff_text:quarterbeats,payload:score_staff_text:mc_onset,payload:score_staff_text:mn_onset,payload:score_staff_text:timesig,payload:score_staff_text:staff,payload:score_staff_text:voice,payload:score_staff_text:duration,payload:score_staff_text:volta,payload:score_staff_text:slur,payload:score_staff_text:staff_text
0,score_chord,0,1,1,1,0,0,0,3/4,1,1,1/4,None,0,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,score_chord,0,1,1,1,0,0,0,3/4,2,1,1/4,None,1,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,score_chord,0,1,1,1,0,0,0,3/4,3,1,1/4,None,2,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,score_chord,0,1,1,1,0,0,0,3/4,4,1,1/4,None,3,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,score_dynamic,0,0,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
5,score_dynamic,0,0,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
6,score_dynamic,0,0,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
7,score_dynamic,0,0,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
8,score_chord,1,1/2,1,1,1,1/4,1/4,3/4,1,1,1/8,None,0,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
9,score_chord,1,1/2,1,1,1,1/4,1/4,3/4,2,1,1/8,None,1,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [6]:
df.iloc[:, :14].notna().groupby(df['payload_type']).mean().round(2)

,payload_type,extent:time_interval:start:quarters,extent:time_interval:duration:quarters,payload:score_chord:mc,payload:score_chord:mn,payload:score_chord:quarterbeats,payload:score_chord:mc_onset,payload:score_chord:mn_onset,payload:score_chord:timesig,payload:score_chord:staff,payload:score_chord:voice,payload:score_chord:duration,payload:score_chord:volta,payload:score_chord:slur
payload_type,,,,,,,,,,,,,,
score_chord,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.25
score_dynamic,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
score_spanner,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
score_staff_text,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00


In [7]:
csv = read_bopp_csv(path.with_suffix('').with_suffix('.bopp.csv'))
print(csv.attrs)
csv[csv['payload_type'] == 'score_dynamic'].dropna(axis=1, how='all').head()

{'object_type': 'score_object', 'media_id': 'dlc:ABC/n01op18-1_01', 'bopp_version': '1.0.0'}


,payload_type,extent:time_interval:start:quarters,extent:time_interval:duration:quarters,payload:score_dynamic:mc,payload:score_dynamic:mn,payload:score_dynamic:quarterbeats,payload:score_dynamic:mc_onset,payload:score_dynamic:mn_onset,payload:score_dynamic:timesig,payload:score_dynamic:staff,payload:score_dynamic:voice,payload:score_dynamic:duration,payload:score_dynamic:slur,payload:score_dynamic:dynamics
4,score_dynamic,0,0,1.0,1.0,0,0,0,3/4,1.0,1.0,0.0,None,p
5,score_dynamic,0,0,1.0,1.0,0,0,0,3/4,2.0,1.0,0.0,None,p
6,score_dynamic,0,0,1.0,1.0,0,0,0,3/4,3.0,1.0,0.0,None,p
7,score_dynamic,0,0,1.0,1.0,0,0,0,3/4,4.0,1.0,0.0,None,p
117,score_dynamic,24,0,9.0,9.0,24,0,0,3/4,1.0,1.0,0.0,None,f
